# Building Agentic AutoML

## Goal

This notebook is a hands-on journey to build an Agentic AutoML system from scratch.

Each version introduces one new concept, allowing the agent to evolve step by step while practicing AutoML and agent development.

---

## Version 7

In this version, we extend the agent with hyperparameter optimization.

The agent now compares multiple hyperparameter configurations for each candidate model while preserving the preprocessing, feature engineering, metric, and validation logic introduced in previous versions.

The search remains intentionally small and controlled so that Version 7 focuses on one new concept only: **Hyperparameter Optimization**.

The rest of the pipeline remains unchanged.

## 1. Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier, CatBoostRegressor
from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor

from sklearn.metrics import mean_squared_error, roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold

## 2. Data

We continue using the Adult Income dataset so that the only new concept introduced in Version 7 is hyperparameter optimization.

Keeping the same dataset allows us to compare tuned model configurations directly with the fixed configurations evaluated in Version 6.

The dataset still provides numerical and categorical features, missing values, and enough complexity to evaluate the effect of model hyperparameters.

The target column is provided by the user.

In [2]:
DATA_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/adult_income.csv"
TARGET = "income"

df = pd.read_csv(DATA_PATH)

df.head()

,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


## 3. Task Detection

The agent first inspects the target to identify the machine learning task.

For now, we use a simple rule based on the number of unique target values.

In [3]:
def detect_task(df, target):
    n_unique = df[target].nunique()

    if n_unique <= 20:
        return "classification"

    return "regression"

In [4]:
task = detect_task(df, TARGET)

task

'classification'

## 4. Feature Detection

The agent identifies numerical and categorical features.

For now, feature types are detected directly from the dataframe dtypes.

In [5]:
def detect_features(df, target):
    X = df.drop(columns=target)

    numerical = X.select_dtypes(include="number").columns.tolist()
    categorical = X.select_dtypes(exclude="number").columns.tolist()

    return numerical, categorical

In [6]:
numerical_features, categorical_features = detect_features(df, TARGET)

numerical_features, categorical_features

(['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'])

## 5. Data Inspection

The agent now inspects the dataset before training.

It summarizes the dataset dimensions, target distribution, feature data types, missing values, and feature cardinality.

In [7]:
def inspect_dataset(df, target):
    X = df.drop(columns=target)
    y = df[target]

    target_info = {
        "name": target,
        "dtype": str(y.dtype),
        "unique_values": int(y.nunique())
    }

    if y.nunique() <= 20:
        target_info["distribution"] = y.value_counts(dropna=False).to_dict()
    else:
        target_info["summary"] = y.describe().to_dict()

    return {
        "shape": {"rows": len(df), "columns": len(df.columns)},
        "target": target_info,
        "dtypes": {column: str(dtype) for column, dtype in X.dtypes.items()},
        "missing_values": df.isna().sum().to_dict(),
        "cardinality": X.nunique(dropna=True).to_dict()
    }

In [8]:
inspection = inspect_dataset(df, TARGET)

inspection

{'shape': {'rows': 48842, 'columns': 15},
 'target': {'name': 'income',
  'dtype': 'object',
  'unique_values': 2,
  'distribution': {'<=50K': 37155, '>50K': 11687}},
 'dtypes': {'age': 'int64',
  'workclass': 'object',
  'fnlwgt': 'int64',
  'education': 'object',
  'education_num': 'int64',
  'marital_status': 'object',
  'occupation': 'object',
  'relationship': 'object',
  'race': 'object',
  'sex': 'object',
  'capital_gain': 'int64',
  'capital_loss': 'int64',
  'hours_per_week': 'int64',
  'native_country': 'object'},
 'missing_values': {'age': 0,
  'workclass': 2799,
  'fnlwgt': 0,
  'education': 0,
  'education_num': 0,
  'marital_status': 0,
  'occupation': 2809,
  'relationship': 0,
  'race': 0,
  'sex': 0,
  'capital_gain': 0,
  'capital_loss': 0,
  'hours_per_week': 0,
  'native_country': 857,
  'income': 0},
 'cardinality': {'age': 74,
  'workclass': 8,
  'fnlwgt': 28523,
  'education': 16,
  'education_num': 16,
  'marital_status': 7,
  'occupation': 14,
  'relationship'

## 6. Preprocessing

The agent tries different preprocessing strategies before training.

We keep the same two approaches introduced in Version 3:

- `native`: preserve missing values whenever the selected model supports them directly
- `impute`: fill missing numerical values with the median and categorical values with the most frequent value

Some models may require minimal technical adaptation of categorical missing values before training.

Preprocessing parameters are learned only from the training data.

In [9]:
def prepare_data(df, target):
    X = df.drop(columns=target).copy()
    y = df[target].copy()

    return X, y

In [10]:
X, y = prepare_data(df, TARGET)

X.shape, y.shape

((48842, 14), (48842,))

In [11]:
def preprocess_data(X_train, X_valid, numerical_features, categorical_features, strategy):
    X_train = X_train.copy()
    X_valid = X_valid.copy()

    if strategy == "impute":
        for column in numerical_features:
            value = X_train[column].median()
            X_train[column] = X_train[column].fillna(value)
            X_valid[column] = X_valid[column].fillna(value)

        for column in categorical_features:
            value = X_train[column].mode().iloc[0]
            X_train[column] = X_train[column].fillna(value)
            X_valid[column] = X_valid[column].fillna(value)

    for column in categorical_features:
        categories = X_train[column].dropna().unique()
        dtype = pd.CategoricalDtype(categories=categories)

        X_train[column] = X_train[column].astype(dtype)
        X_valid[column] = X_valid[column].astype(dtype)

    return X_train, X_valid

In [12]:
PREPROCESSING_STRATEGIES = ["native", "impute"]

PREPROCESSING_STRATEGIES

['native', 'impute']

## 7. Feature Engineering

The feature engineering layer introduced in Version 6 remains unchanged.

The agent compares two feature engineering strategies:

- `none`: keep the original feature space unchanged
- `interactions`: add pairwise multiplication features between numerical variables

Feature engineering is applied independently inside each cross-validation fold after preprocessing.

The original features are always preserved.

Keeping feature engineering unchanged allows Version 7 to isolate the effect of hyperparameter optimization.

In [13]:
FEATURE_ENGINEERING_STRATEGIES = ["none", "interactions"]


def apply_feature_engineering(X_train, X_valid, numerical_features, strategy):
    X_train = X_train.copy()
    X_valid = X_valid.copy()

    if strategy == "interactions":
        for i in range(len(numerical_features)):
            for j in range(i + 1, len(numerical_features)):
                feature_a = numerical_features[i]
                feature_b = numerical_features[j]

                new_feature = f"{feature_a}_x_{feature_b}"

                X_train[new_feature] = (X_train[feature_a] * X_train[feature_b])

                X_valid[new_feature] = (X_valid[feature_a] * X_valid[feature_b])

    return X_train, X_valid

## 8. Model Selection

The agent compares the same three boosting model families used in Version 6:

- LightGBM
- XGBoost
- CatBoost

The model candidates remain unchanged so that Version 7 can isolate the effect of hyperparameter optimization.

Unlike previous versions, each model will now be evaluated under multiple hyperparameter configurations.

In [14]:
def select_models(task, categorical_features, model_params=None):
    model_params = model_params or {}

    lightgbm_params = model_params.get("LightGBM", {})
    xgboost_params = model_params.get("XGBoost", {})
    catboost_params = model_params.get("CatBoost", {})

    if task == "classification":
        return {
            "LightGBM": LGBMClassifier(
                random_state=42,
                verbosity=-1,
                **lightgbm_params
            ),
            "XGBoost": XGBClassifier(
                random_state=42,
                tree_method="hist",
                enable_categorical=True,
                verbosity=0,
                **xgboost_params
            ),
            "CatBoost": CatBoostClassifier(
                random_seed=42,
                cat_features=categorical_features,
                verbose=False,
                allow_writing_files=False,
                **catboost_params
            )
        }

    return {
        "LightGBM": LGBMRegressor(
            random_state=42,
            verbosity=-1,
            **lightgbm_params
        ),
        "XGBoost": XGBRegressor(
            random_state=42,
            tree_method="hist",
            enable_categorical=True,
            verbosity=0,
            **xgboost_params
        ),
        "CatBoost": CatBoostRegressor(
            random_seed=42,
            cat_features=categorical_features,
            verbose=False,
            allow_writing_files=False,
            **catboost_params
        )
    }

In [15]:
models = select_models(task, categorical_features)

models

{'LightGBM': LGBMClassifier(random_state=42, verbosity=-1),
 'XGBoost': XGBClassifier(base_score=None, booster=None, callbacks=None,
               colsample_bylevel=None, colsample_bynode=None,
               colsample_bytree=None, device=None, early_stopping_rounds=None,
               enable_categorical=True, eval_metric=None, feature_types=None,
               feature_weights=None, gamma=None, grow_policy=None,
               importance_type=None, interaction_constraints=None,
               learning_rate=None, max_bin=None, max_cat_threshold=None,
               max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
               max_leaves=None, min_child_weight=None, missing=nan,
               monotone_constraints=None, multi_strategy=None, n_estimators=None,
               n_jobs=None, num_parallel_tree=None, ...),
 'CatBoost': CatBoostClassifier(allow_writing_files=False, cat_features=['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 's

## 9. Hyperparameter Optimization

Version 7 introduces hyperparameter optimization.

For each candidate model, the agent compares a small set of predefined hyperparameter configurations.

The search space remains intentionally compact so that the effect of hyperparameter tuning can be studied without introducing a more complex optimization framework.

Each model keeps its original Version 6 configuration as the baseline and compares it with one alternative configuration.

Hyperparameter configurations are evaluated using the same preprocessing strategies, feature engineering strategies, metrics, and cross-validation protocol introduced in previous versions.

In [16]:
HYPERPARAMETER_SEARCH_SPACES = {
    "LightGBM": [
        {},
        {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "num_leaves": 31
        }
    ],
    "XGBoost": [
        {},
        {
            "n_estimators": 200,
            "learning_rate": 0.05,
            "max_depth": 4
        }
    ],
    "CatBoost": [
        {},
        {
            "iterations": 500,
            "learning_rate": 0.05,
            "depth": 6
        }
    ]
}

HYPERPARAMETER_SEARCH_SPACES

{'LightGBM': [{},
  {'n_estimators': 200, 'learning_rate': 0.05, 'num_leaves': 31}],
 'XGBoost': [{}, {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4}],
 'CatBoost': [{}, {'iterations': 500, 'learning_rate': 0.05, 'depth': 6}]}

## 10. Metric Selection

The evaluation metric remains unchanged from previous versions.

The agent uses:

- `ROC AUC` for classification
- `RMSE` for regression

Keeping the metric logic unchanged allows Version 7 to isolate the effect of hyperparameter optimization.

In [17]:
def select_metric(task):
    if task == "classification":
        return "roc_auc"

    return "rmse"

In [18]:
metric = select_metric(task)

metric

'roc_auc'

## 11. Validation Strategy

The smart validation strategy introduced in Version 5 remains unchanged.

The agent uses:

- `StratifiedKFold` for classification
- `KFold` for regression

Both strategies use 5 folds, `shuffle=True`, and `random_state=42`.

Keeping validation unchanged allows Version 7 to isolate the effect of hyperparameter optimization.

In [19]:
def select_validation(task, n_splits=5):
    if task == "classification":
        return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

    return KFold(n_splits=n_splits, shuffle=True, random_state=42)

In [20]:
validation = select_validation(task)

validation

StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

## 12. Training and Evaluation

The agent now evaluates every preprocessing-feature engineering-model-hyperparameter combination across all cross-validation folds.

Inside each fold, preprocessing is applied first, feature engineering is applied immediately afterwards, and a fresh model instance is created using the hyperparameter configuration being evaluated.

For every experiment, the agent stores the selected hyperparameters, the score obtained on each fold, the mean score, and the standard deviation.

The best experiment is selected using the mean validation score.

In [21]:
def train_and_evaluate(
    X,
    y,
    model_name,
    task,
    numerical_features,
    categorical_features,
    preprocessing,
    feature_engineering,
    model_params,
    validation
):
    fold_scores = []

    for train_idx, valid_idx in validation.split(X, y):
        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()

        y_train = y.iloc[train_idx].copy()
        y_valid = y.iloc[valid_idx].copy()

        X_train, X_valid = preprocess_data(
            X_train,
            X_valid,
            numerical_features,
            categorical_features,
            preprocessing
        )

        X_train, X_valid = apply_feature_engineering(
            X_train,
            X_valid,
            numerical_features,
            feature_engineering
        )

        fold_model = select_models(
            task,
            categorical_features,
            {
                model_name: model_params
            }
        )[model_name]

        y_fit = y_train
        y_eval = y_valid

        if model_name == "CatBoost":
            for column in categorical_features:
                X_train[column] = (
                    X_train[column]
                    .astype("object")
                    .fillna("__MISSING__")
                    .astype(str)
                )

                X_valid[column] = (
                    X_valid[column]
                    .astype("object")
                    .fillna("__MISSING__")
                    .astype(str)
                )

        if model_name == "XGBoost" and task == "classification":
            classes = sorted(y_train.unique())

            mapping = {label: index for index, label in enumerate(classes)}

            y_fit = y_train.map(mapping)
            y_eval = y_valid.map(mapping)

        fold_model.fit(X_train, y_fit)

        if task == "classification":
            predictions = fold_model.predict_proba(X_valid)[:, 1]

            score = roc_auc_score(y_eval, predictions)

        else:
            predictions = fold_model.predict(X_valid)

            score = mean_squared_error(y_valid, predictions) ** 0.5

        fold_scores.append(float(score))

    return {
        "fold_scores": fold_scores,
        "mean_score": float(np.mean(fold_scores)),
        "std_score": float(np.std(fold_scores))
    }

In [22]:
experiments = []

for preprocessing in PREPROCESSING_STRATEGIES:
    for feature_engineering in FEATURE_ENGINEERING_STRATEGIES:
        for model_name in models:
            for model_params in HYPERPARAMETER_SEARCH_SPACES[model_name]:
                result = train_and_evaluate(
                    X,
                    y,
                    model_name,
                    task,
                    numerical_features,
                    categorical_features,
                    preprocessing,
                    feature_engineering,
                    model_params,
                    validation
                )

                experiments.append({
                    "preprocessing": preprocessing,
                    "feature_engineering": feature_engineering,
                    "model": model_name,
                    "hyperparameters": model_params,
                    "fold_scores": result["fold_scores"],
                    "mean_score": result["mean_score"],
                    "std_score": result["std_score"]
                })

experiments

[{'preprocessing': 'native',
  'feature_engineering': 'none',
  'model': 'LightGBM',
  'hyperparameters': {},
  'fold_scores': [0.9256299673563653,
   0.930524872165813,
   0.9315811297628095,
   0.9323195737110039,
   0.9277226104177834],
  'mean_score': 0.9295556306827549,
  'std_score': 0.0025080770515734153},
 {'preprocessing': 'native',
  'feature_engineering': 'none',
  'model': 'LightGBM',
  'hyperparameters': {'n_estimators': 200,
   'learning_rate': 0.05,
   'num_leaves': 31},
  'fold_scores': [0.9258859004984437,
   0.9306843950946944,
   0.9318443990805844,
   0.9333074670652791,
   0.9275074228761112],
  'mean_score': 0.9298459169230225,
  'std_score': 0.0027506214009272102},
 {'preprocessing': 'native',
  'feature_engineering': 'none',
  'model': 'XGBoost',
  'hyperparameters': {},
  'fold_scores': [0.92372150560175,
   0.927186891572412,
   0.9290222291552113,
   0.9293962593069186,
   0.9249302108855184],
  'mean_score': 0.9268514193043622,
  'std_score': 0.0022265924307

In [23]:
def select_best_experiment(experiments, metric):
    if metric == "rmse":
        return min(experiments, key=lambda experiment: experiment["mean_score"])

    return max(experiments, key=lambda experiment: experiment["mean_score"])

In [24]:
best_experiment = select_best_experiment(experiments, metric)

state = {
    "task": task,
    "metric": metric,
    "validation": validation.__class__.__name__,
    "n_splits": validation.n_splits,
    "numerical_features": numerical_features,
    "categorical_features": categorical_features,
    "best_preprocessing": best_experiment["preprocessing"],
    "best_feature_engineering": best_experiment["feature_engineering"],
    "best_model": best_experiment["model"],
    "best_params": best_experiment["hyperparameters"],
    "best_score": best_experiment["mean_score"],
    "best_std": best_experiment["std_score"],
    "experiments": experiments
}

state

{'task': 'classification',
 'metric': 'roc_auc',
 'validation': 'StratifiedKFold',
 'n_splits': 5,
 'numerical_features': ['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 'categorical_features': ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'],
 'best_preprocessing': 'native',
 'best_feature_engineering': 'none',
 'best_model': 'CatBoost',
 'best_params': {},
 'best_score': 0.930630626301291,
 'best_std': 0.002246310473215999,
 'experiments': [{'preprocessing': 'native',
   'feature_engineering': 'none',
   'model': 'LightGBM',
   'hyperparameters': {},
   'fold_scores': [0.9256299673563653,
    0.930524872165813,
    0.9315811297628095,
    0.9323195737110039,
    0.9277226104177834],
   'mean_score': 0.9295556306827549,
   'std_score': 0.0025080770515734153},
  {'preprocessing': 'native',
   'feature_engineering': 'none',
   'model': 'LightGBM',
   'hyperparameters

## 13. Agent

We now combine the previous components into a single agent.

The agent inspects the dataset, selects the validation strategy, compares preprocessing strategies, compares feature engineering strategies, evaluates multiple hyperparameter configurations for every candidate model across the cross-validation folds, and selects the best experiment using the mean validation score.

Version 7 therefore extends the experiment space without changing the previous decision logic.

In [25]:
def agent(data_path, target):
    df = pd.read_csv(data_path)

    task = detect_task(df, target)
    numerical, categorical = detect_features(df, target)
    inspection = inspect_dataset(df, target)
    X, y = prepare_data(df, target)

    metric = select_metric(task)
    validation = select_validation(task)
    models = select_models(task, categorical)

    experiments = []

    for preprocessing in PREPROCESSING_STRATEGIES:
        for feature_engineering in FEATURE_ENGINEERING_STRATEGIES:
            for model_name in models:
                for model_params in HYPERPARAMETER_SEARCH_SPACES[model_name]:
                    result = train_and_evaluate(
                        X,
                        y,
                        model_name,
                        task,
                        numerical,
                        categorical,
                        preprocessing,
                        feature_engineering,
                        model_params,
                        validation
                    )

                    experiments.append({
                        "preprocessing": preprocessing,
                        "feature_engineering": feature_engineering,
                        "model": model_name,
                        "hyperparameters": model_params,
                        "fold_scores": result["fold_scores"],
                        "mean_score": result["mean_score"],
                        "std_score": result["std_score"]
                    })

    best_experiment = select_best_experiment(experiments, metric)

    return {
        "task": task,
        "numerical_features": numerical,
        "categorical_features": categorical,
        "inspection": inspection,
        "metric": metric,
        "validation": validation.__class__.__name__,
        "n_splits": validation.n_splits,
        "experiments": experiments,
        "best_preprocessing": best_experiment["preprocessing"],
        "best_feature_engineering": best_experiment["feature_engineering"],
        "best_model": best_experiment["model"],
        "best_params": best_experiment["hyperparameters"],
        "best_score": best_experiment["mean_score"],
        "best_std": best_experiment["std_score"]
    }

In [26]:
state = agent(DATA_PATH, TARGET)

state

{'task': 'classification',
 'numerical_features': ['age',
  'fnlwgt',
  'education_num',
  'capital_gain',
  'capital_loss',
  'hours_per_week'],
 'categorical_features': ['workclass',
  'education',
  'marital_status',
  'occupation',
  'relationship',
  'race',
  'sex',
  'native_country'],
 'inspection': {'shape': {'rows': 48842, 'columns': 15},
  'target': {'name': 'income',
   'dtype': 'object',
   'unique_values': 2,
   'distribution': {'<=50K': 37155, '>50K': 11687}},
  'dtypes': {'age': 'int64',
   'workclass': 'object',
   'fnlwgt': 'int64',
   'education': 'object',
   'education_num': 'int64',
   'marital_status': 'object',
   'occupation': 'object',
   'relationship': 'object',
   'race': 'object',
   'sex': 'object',
   'capital_gain': 'int64',
   'capital_loss': 'int64',
   'hours_per_week': 'int64',
   'native_country': 'object'},
  'missing_values': {'age': 0,
   'workclass': 2799,
   'fnlwgt': 0,
   'education': 0,
   'education_num': 0,
   'marital_status': 0,
   'occ

## 14. Regression Test

The same agent should also work with a regression dataset.

We only change the input dataset and target column.

The agent must detect regression automatically, use `KFold` validation, compare all preprocessing-feature engineering-model-hyperparameter combinations across 5 folds, and select the best experiment using the mean RMSE.

This verifies that the hyperparameter optimization logic works consistently for both classification and regression.

In [27]:
REGRESSION_PATH = "/kaggle/input/datasets/lucalullo/agentic-automl-datasets/simple_regression.csv"

regression_state = agent(REGRESSION_PATH, "target")

regression_state

{'task': 'regression',
 'numerical_features': ['num_1', 'num_2', 'num_3', 'num_4'],
 'categorical_features': ['category_1', 'category_2'],
 'inspection': {'shape': {'rows': 1200, 'columns': 7},
  'target': {'name': 'target',
   'dtype': 'float64',
   'unique_values': 1200,
   'summary': {'count': 1200.0,
    'mean': 1.4709987021728987,
    'std': 5.814924591631912,
    'min': -16.008721903706515,
    '25%': -2.555178021015151,
    '50%': 1.3890592022588901,
    '75%': 5.322514631724257,
    'max': 24.027912326916383}},
  'dtypes': {'num_1': 'float64',
   'num_2': 'float64',
   'num_3': 'float64',
   'num_4': 'float64',
   'category_1': 'object',
   'category_2': 'object'},
  'missing_values': {'num_1': 9,
   'num_2': 8,
   'num_3': 10,
   'num_4': 4,
   'category_1': 9,
   'category_2': 20,
   'target': 0},
  'cardinality': {'num_1': 1191,
   'num_2': 1192,
   'num_3': 1190,
   'num_4': 1196,
   'category_1': 3,
   'category_2': 2}},
 'metric': 'rmse',
 'validation': 'KFold',
 'n_split

## Notes

The agent now:

- detects the machine learning task;
- inspects the dataset;
- compares the available preprocessing strategies;
- compares the available feature engineering strategies;
- compares LightGBM, XGBoost, and CatBoost;
- evaluates multiple hyperparameter configurations for each model;
- selects the validation strategy according to the detected task;
- uses `StratifiedKFold` for classification and `KFold` for regression;
- evaluates every preprocessing-feature engineering-model-hyperparameter combination across 5 folds;
- stores the hyperparameters, fold scores, mean score, and standard deviation for every experiment;
- selects the best experiment using the mean validation score;
- returns the best preprocessing strategy, feature engineering strategy, model, hyperparameters, score, and score variability.

Version 7 uses a deliberately small hyperparameter search space.

Each model compares its original configuration with one alternative configuration.

With 2 preprocessing strategies, 2 feature engineering strategies, 3 models, 2 hyperparameter configurations, and 5 folds, the agent performs 120 model fits per dataset.

Preprocessing and feature engineering remain inside each cross-validation fold.

The architecture remains intentionally simple.

Future versions will introduce new components and gradually evolve the architecture.